# LogiTrack Predictive Analytics

This Colab-ready notebook develops and evaluates models that predict delivery duration in minutes from historical logistics characteristics.

## 1. Setup

The notebook uses fixed random seeds. Upload `amazon_delivery.csv` to the Colab session before running the data-loading cell.

In [ ]:
!pip -q install pandas numpy scikit-learn joblib matplotlib seaborn
import pandas as pd, numpy as np, joblib
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.dummy import DummyRegressor
from sklearn.linear_model import Ridge
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
SEED=42

## 2. Data loading and inspection

In [ ]:
df=pd.read_csv('amazon_delivery.csv')
print(df.shape)
display(df.head())
display(df.dtypes)
display(df.isna().sum())
print('Exact duplicate rows:',df.duplicated().sum())

## 3. Exploratory analysis

Inspect the delivery-time target, categorical comparisons, missingness, and coordinate quality before choosing features.

In [ ]:
display(df['Delivery_Time'].describe())
display(df.groupby('Traffic')['Delivery_Time'].agg(['count','mean','median']).sort_values('mean',ascending=False))
display(df.groupby('Weather')['Delivery_Time'].agg(['count','mean','median']).sort_values('mean',ascending=False))

## 4. Feature engineering and leakage control

`Pickup_Time` is excluded because it can occur after order placement. Haversine distance uses valid coordinates. The distance cap is learned from training data only.

In [ ]:
def haversine(lat1,lon1,lat2,lon2):
 r=6371.; p1,p2=np.radians(lat1),np.radians(lat2); dp=np.radians(lat2-lat1); dl=np.radians(lon2-lon1); a=np.sin(dp/2)**2+np.cos(p1)*np.cos(p2)*np.sin(dl/2)**2; return 2*r*np.arcsin(np.sqrt(np.clip(a,0,1)))
df['Order_Date']=pd.to_datetime(df['Order_Date'],errors='coerce')
for c in ['Agent_Age','Agent_Rating','Store_Latitude','Store_Longitude','Drop_Latitude','Drop_Longitude','Delivery_Time']: df[c]=pd.to_numeric(df[c],errors='coerce')
valid=df[['Store_Latitude','Store_Longitude','Drop_Latitude','Drop_Longitude']].notna().all(axis=1)&df.Store_Latitude.between(-90,90)&df.Drop_Latitude.between(-90,90)&df.Store_Longitude.between(-180,180)&df.Drop_Longitude.between(-180,180)
df=df.loc[valid&df.Order_Date.notna()&df.Delivery_Time.notna()].copy()
df['Distance_km']=haversine(df.Store_Latitude,df.Store_Longitude,df.Drop_Latitude,df.Drop_Longitude)
df['Order_Hour']=pd.to_datetime(df.Order_Time,format='%H:%M:%S',errors='coerce').dt.hour.fillna(-1)
df['Day_of_Week']=df.Order_Date.dt.dayofweek
for c in ['Weather','Traffic','Vehicle','Area','Category']: df[c]=df[c].fillna('Unknown').astype(str).str.strip().replace({'NaN':'Unknown','nan':'Unknown','':'Unknown'})

## 5. Chronological split and preprocessing

In [ ]:
df=df.sort_values('Order_Date').reset_index(drop=True)
split_date=df.Order_Date.iloc[int(len(df)*.8)]
train,test=df[df.Order_Date<split_date].copy(),df[df.Order_Date>=split_date].copy()
cap=train.Distance_km.quantile(.99)
for x in [train,test]: x['Distance_Model_km']=x.Distance_km.clip(upper=cap)
num=['Agent_Age','Agent_Rating','Distance_Model_km','Order_Hour','Day_of_Week']; cat=['Weather','Traffic','Vehicle','Area','Category']; features=num+cat
pre=ColumnTransformer([('num',Pipeline([('impute',SimpleImputer(strategy='median')),('scale',StandardScaler())]),num),('cat',Pipeline([('impute',SimpleImputer(strategy='most_frequent')),('onehot',OneHotEncoder(handle_unknown='ignore',sparse_output=False))]),cat)])

## 6. Candidate models and held-out evaluation

In [ ]:
models={'Dummy':DummyRegressor(),'Ridge':Pipeline([('pre',pre),('m',Ridge(alpha=3))]),'Random Forest':Pipeline([('pre',pre),('m',RandomForestRegressor(n_estimators=160,min_samples_leaf=3,max_features=.8,n_jobs=-1,random_state=SEED))]),'Gradient Boosting':Pipeline([('pre',pre),('m',GradientBoostingRegressor(n_estimators=220,learning_rate=.045,max_depth=3,min_samples_leaf=8,random_state=SEED))])}
results=[]; fitted={}
for name,m in models.items():
 m.fit(train[features],train.Delivery_Time); p=m.predict(test[features]); results.append([name,mean_absolute_error(test.Delivery_Time,p),mean_squared_error(test.Delivery_Time,p)**.5,r2_score(test.Delivery_Time,p)]); fitted[name]=m
results=pd.DataFrame(results,columns=['Model','MAE','RMSE','R2']).sort_values('MAE'); display(results)

## 7. Diagnostics, interpretation, and export

Select the best model by validation criteria without tuning on the untouched test set in later refinements. Importance measures predictive contribution, not causation.

In [ ]:
best_name=results.iloc[0].Model; best=fitted[best_name]; pred=best.predict(test[features])
import matplotlib.pyplot as plt
plt.scatter(test.Delivery_Time,pred,alpha=.15); plt.xlabel('Actual minutes'); plt.ylabel('Predicted minutes'); plt.title('Held-out actual vs predicted'); plt.show()
plt.hist(test.Delivery_Time-pred,bins=40); plt.title('Held-out residual distribution'); plt.show()
joblib.dump(best,'delivery_time_pipeline.joblib')
print('Selected model:',best_name,'Distance cap:',cap)

## 8. Conclusion

Report only metrics produced by the executed notebook. Explain geographical anomalies, model uncertainty, the historical scope of the data, and the lack of sufficient date coverage for demand forecasting.